# Example 1.3.2 / 3: Initial Values from the Data

Companion notebook for **1.3.2 Defined Variables and Initial Values** in the AMPL lab Documentation, section "Initial Values from the Data". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Initial Values from the Data

Initial values can also come from a data section, `var Make := bands 100 coils 200;`, for
variables declared without `:=` (with `default`, or with neither). From Python,
`set_value(v)` on a scalar variable or one instance, and `set_values({...})` on an indexed
variable, set the current values instead, like `let`; they don't fix the variable, so the
solve still replaces them:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param rate {PROD} > 0;
param profit {PROD};
param market {PROD} >= 0;
param avail >= 0;

var Make {p in PROD} >= 0, <= market[p];

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}
ampl.var["Make"].set_values({"bands": 100, "coils": 200})

In [5]:
# run.py
print(ampl.var["Make"].to_dict())   # before solving: the values just set
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())   # after solving

{'bands': 100, 'coils': 200}
HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 1400}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; the data file gives the initial values with a
`var` statement:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    30     4000 ;
var Make := bands 100 coils 200;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display Make;         # before solving: the initial values
option solver highs;
solve;
display Make;         # after solving

Make [*] :=
bands  100
coils  200
;

HiGHS 1.14.0: optimal solution; objective 192000
0 simplex iterations
0 barrier iterations
Make [*] :=
bands  6000
coils  1400
;

